In [ ]:
import duckdb
import pandas as pd
import numpy as np
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from networks.feed_forward_tabular import NeuralModel
from networks.feed_forward_tabular import build_dataloaders

con = duckdb.connect('../capillary.db')

df = con.execute(""" SELECT row_id, value, label, albumin,antitrypsin,orosomukoid,haptoglobin,crp,igg,iga,igm,set FROM protein_data WHERE value IS NOT NULL
                 AND observation_nr = 1
                 AND analysis IS NOT NULL
                 AND protein_value IS NOT NULL""").df()

con.close()


print(f"Antal fall med M-komponent:{(df['label'] == 1).sum()}")
print(f"Antal fall utan M-komponent:{(df['label'] == 0).sum()}")


train_rows = df[df['set'] == 'train']
val_rows = df[df['set'] == 'val']
test_rows = df[df['set'] == 'test']

drop_indices = train_rows[train_rows['label'] == 0].sample(frac=0.7).index


train_rows = train_rows.drop(drop_indices)
train_rows = train_rows[train_rows['label'].isin([0,1])]
val_rows = val_rows[val_rows['label'].isin([0,1])]

print(f"Antal utan m-komponent i träningsdatan: {len(train_rows[train_rows['label'] == 0])}")
print(f"Antal med m-komponent i träningsdatan: {len(train_rows[train_rows['label'] == 1])}")

network = NeuralModel()
network.reset_weights()
cnn_train_dl, cnn_val_dl, _ = build_dataloaders(train_rows, val_rows, val_rows)
network.retrain(cnn_train_dl,cnn_val_dl,patience=15)

Antal fall med M-komponent:2942
Antal fall utan M-komponent:69882
Antal utan m-komponent i träningsdatan: 17903
Antal med m-komponent i träningsdatan: 2553
Total parameters: 251,970
  -> ny bästa modell sparad till ../models/feed_forward_tabular.pth
Epoch   0 | train: 1.2580 | val: 0.4902 | acc: 90.51% | AUC: 0.764  | LR: 0.001
  -> ny bästa modell sparad till ../models/feed_forward_tabular.pth
Epoch   1 | train: 0.5559 | val: 0.4018 | acc: 95.86% | AUC: 0.839  | LR: 0.001
  -> ny bästa modell sparad till ../models/feed_forward_tabular.pth
Epoch   2 | train: 0.4890 | val: 0.3472 | acc: 96.08% | AUC: 0.855  | LR: 0.001
  -> ny bästa modell sparad till ../models/feed_forward_tabular.pth
Epoch   3 | train: 0.5091 | val: 0.5114 | acc: 78.37% | AUC: 0.886  | LR: 0.001
  -> ny bästa modell sparad till ../models/feed_forward_tabular.pth
Epoch   4 | train: 0.4508 | val: 0.3020 | acc: 96.41% | AUC: 0.911  | LR: 0.001
  -> ny bästa modell sparad till ../models/feed_forward_tabular.pth
Epoch   5 

In [ ]:
con = duckdb.connect('../capillary.db')

df = con.execute(""" SELECT row_id, value, label, albumin,antitrypsin,orosomukoid,haptoglobin,crp,igg,iga,igm,set FROM protein_data WHERE value IS NOT NULL
                 AND observation_nr = 1
                 AND analysis IS NOT NULL
                 AND protein_value IS NOT NULL""").df()

con.close()


k_fold_rows = df[df['set'].isin(['train','val']) ].copy()
k_fold_rows = k_fold_rows[k_fold_rows['label'].isin([0,1])]
drop_indices = k_fold_rows[k_fold_rows['label'] == 0].sample(frac=0.7).index
k_fold_rows = k_fold_rows.drop(drop_indices)
print(f"Totalt antal utan m-komponent i K-fold poolen: {len(k_fold_rows[k_fold_rows['label'] == 0])}")
print(f"Totalt antal med m-komponent i K-fold poolen: {len(k_fold_rows[k_fold_rows['label'] == 1])}")

test_rows  = df[df['set'] == 'test']


network = NeuralModel()
network.retrain_with_k_fold(k_fold_rows)

Totalt antal utan m-komponent i K-fold poolen: 18848
Totalt antal med m-komponent i K-fold poolen: 2692
Total parameters: 251,970
--- Startar 10-Fold Cross Validation ---

 FOLD 1/10
  -> ny bästa modell sparad till ../models/feed_forward_tabular_fold1.pth
Epoch   0 | train: 1.1368 | val: 0.6286 | acc: 64.39% | AUC: 0.713  | LR: 0.001
  -> ny bästa modell sparad till ../models/feed_forward_tabular_fold1.pth
Epoch   1 | train: 0.5704 | val: 0.5433 | acc: 82.73% | AUC: 0.774  | LR: 0.001
  -> ny bästa modell sparad till ../models/feed_forward_tabular_fold1.pth
Epoch   2 | train: 0.4954 | val: 0.5167 | acc: 90.95% | AUC: 0.819  | LR: 0.001
  -> ny bästa modell sparad till ../models/feed_forward_tabular_fold1.pth
Epoch   3 | train: 0.4821 | val: 0.4932 | acc: 89.23% | AUC: 0.834  | LR: 0.001
Epoch   4 | train: 0.4639 | val: 0.5880 | acc: 47.40% | AUC: 0.783  | LR: 0.001
  -> ny bästa modell sparad till ../models/feed_forward_tabular_fold1.pth
Epoch   5 | train: 0.4524 | val: 0.4771 | acc: 

In [ ]:
from functions.evaluation import evaluate


test_rows = test_rows[test_rows['label'].isin([0,1])]
test_rows['cnn_probability'] = test_rows['probability'].copy()
result = network.predict(test_rows)
_ = evaluate(result)

KeyError: 'probability'